In [6]:
%pip install -q xgboost tensorflow kagglehub scikit-learn numpy pandas imbalanced-learn

In [7]:
from pathlib import Path
import shutil

import kagglehub

# Download latest version
path = kagglehub.dataset_download("muhammadshahidazeem/customer-churn-dataset")

download_dir = Path(path)
print("Path to dataset files:", path)

Using Colab cache for faster access to the 'customer-churn-dataset' dataset.
Path to dataset files: /kaggle/input/customer-churn-dataset


In [10]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Locate the downloaded CSV from kagglehub
csv_candidates = sorted(download_dir.glob("*.csv"))
if not csv_candidates:
    raise FileNotFoundError(f"No CSV files found in {download_dir}")

raw_path = csv_candidates[0]
df = pd.read_csv(raw_path)
print(f"Loaded raw dataset from: {raw_path}")
print(f"Shape: {df.shape}")


def preprocess_frame(frame):
    frame = frame.copy()

    if "CustomerID" in frame.columns:
        frame = frame.drop(columns=["CustomerID"])

    numeric_cols = ["Age", "Tenure", "Usage Frequency", "Support Calls", "Payment Delay", "Total Spend", "Last Interaction"]
    for col in numeric_cols:
        frame[col] = pd.to_numeric(frame[col], errors="coerce")
        frame[col] = frame[col].fillna(frame[col].median())

    categorical_cols = ["Gender", "Subscription Type", "Contract Length"]
    for col in categorical_cols:
        frame[col] = frame[col].fillna(frame[col].mode()[0]).astype(str).str.strip().str.title()

    frame = frame.drop_duplicates()
    frame = frame[(frame["Age"] > 0) & (frame["Tenure"] >= 0) & (frame["Support Calls"] >= 0)]
    frame = frame[(frame["Payment Delay"] >= 0) & (frame["Total Spend"] >= 0) & (frame["Last Interaction"] >= 0)]
    frame = frame[frame["Usage Frequency"] >= 0]

    frame["spend_per_tenure"] = frame["Total Spend"] / (frame["Tenure"] + 1)
    frame["support_call_rate"] = frame["Support Calls"] / (frame["Tenure"] + 1)
    frame["usage_per_tenure"] = frame["Usage Frequency"] / (frame["Tenure"] + 1)
    frame["is_dormant"] = (frame["Last Interaction"] > 30).astype(int)
    frame["has_payment_issues"] = (frame["Payment Delay"] > 0).astype(int)
    frame["spend_tier"] = pd.cut(frame["Total Spend"], bins=[0, 200, 500, float("inf")], labels=[0, 1, 2]).astype(int)
    frame["age_group"] = pd.cut(frame["Age"], bins=[0, 25, 40, 60, float("inf")], labels=[0, 1, 2, 3]).astype(int)

    frame["Gender"] = frame["Gender"].map({"Male": 0, "Female": 1})
    frame = pd.get_dummies(frame, columns=["Subscription Type"], prefix="sub", drop_first=False)
    frame = pd.get_dummies(frame, columns=["Contract Length"], prefix="contract", drop_first=False)
    frame = frame.dropna(subset=["Gender", "Churn"])
    return frame


def split_and_scale_and_balance(frame):
    frame = frame.copy()
    X = frame.drop(columns=["Churn"])
    y = frame["Churn"].astype(int)

    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=0.30, stratify=y, random_state=42
    )
    X_cv, X_test, y_cv, y_test = train_test_split(
        X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42
    )

    scaler = StandardScaler()
    scale_cols = [
        "Age", "Tenure", "Usage Frequency", "Support Calls", "Payment Delay",
        "Total Spend", "Last Interaction", "spend_per_tenure", "support_call_rate", "usage_per_tenure"
    ]
    scale_cols = [c for c in scale_cols if c in X_train.columns]

    X_train[scale_cols] = scaler.fit_transform(X_train[scale_cols])
    X_cv[scale_cols] = scaler.transform(X_cv[scale_cols])
    X_test[scale_cols] = scaler.transform(X_test[scale_cols])

    smote = SMOTE(random_state=42)
    X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

    return X_train, X_cv, X_test, y_train, y_cv, y_test, X_train_bal, y_train_bal


df = preprocess_frame(df)
X_train, X_cv, X_test, y_train, y_cv, y_test, X_train_bal, y_train_bal = split_and_scale_and_balance(df)

print(f"Processed shape: {df.shape}")
print(f"Train shape: {X_train.shape}, CV shape: {X_cv.shape}, Test shape: {X_test.shape}")
print(f"Balanced train shape: {X_train_bal.shape}")

Loaded raw dataset from: /kaggle/input/customer-churn-dataset/customer_churn_dataset-testing-master.csv
Shape: (64374, 12)
Processed shape: (64374, 22)
Train shape: (45061, 21), CV shape: (9656, 21), Test shape: (9657, 21)
Balanced train shape: (47432, 21)


In [13]:
xgb_grid = []
learning_rates = [0.01, 0.03, 0.05, 0.1]
max_depths = [3, 4, 5]
n_estimators_list = [200, 300]

for lr in learning_rates:
    for depth in max_depths:
        for n_estimators in n_estimators_list:
            xgb = XGBClassifier(
                n_estimators=n_estimators,
                max_depth=depth,
                learning_rate=lr,
                subsample=0.9,
                colsample_bytree=0.9,
                eval_metric="logloss",
                random_state=42,
                n_jobs=-1,
            )

            xgb.fit(X_train_bal, y_train_bal)
            xgb_cv_proba = xgb.predict_proba(X_cv)[:, 1]
            xgb_cv_pred = (xgb_cv_proba >= 0.5).astype(int)

            metrics = {
                "learning_rate": lr,
                "max_depth": depth,
                "n_estimators": n_estimators,
                "accuracy": accuracy_score(y_cv, xgb_cv_pred),
                "precision": precision_score(y_cv, xgb_cv_pred, zero_division=0),
                "recall": recall_score(y_cv, xgb_cv_pred, zero_division=0),
                "f1": f1_score(y_cv, xgb_cv_pred, zero_division=0),
                "roc_auc": roc_auc_score(y_cv, xgb_cv_proba),
            }
            xgb_grid.append((metrics, xgb))

xgb_results = pd.DataFrame([m for m, _ in xgb_grid]).sort_values("roc_auc", ascending=False)
print("XGBoost validation results:")
print(xgb_results)

best_xgb_metrics, best_xgb_model = max(xgb_grid, key=lambda item: item[0]["roc_auc"])
print("\nBest XGBoost params:")
print(best_xgb_metrics)


XGBoost validation results:
    learning_rate  max_depth  n_estimators  ...    recall        f1   roc_auc
23           0.10          5           300  ...  1.000000  1.000000  1.000000
22           0.10          5           200  ...  1.000000  0.999891  1.000000
21           0.10          4           300  ...  1.000000  1.000000  1.000000
20           0.10          4           200  ...  1.000000  1.000000  1.000000
15           0.05          4           300  ...  1.000000  0.999891  1.000000
19           0.10          3           300  ...  1.000000  0.999891  1.000000
17           0.05          5           300  ...  1.000000  0.999781  1.000000
11           0.03          5           300  ...  0.999781  0.999454  0.999999
16           0.05          5           200  ...  0.999563  0.999235  0.999998
18           0.10          3           200  ...  0.998688  0.998907  0.999997
14           0.05          4           200  ...  0.997595  0.998359  0.999996
9            0.03          4        

In [12]:
learning_rates = [0.01, 0.001, 0.0001, 0.005, 0.0005]
results = []


def build_nn(input_dim, learning_rate):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(64, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(32, activation="relu"),
        layers.Dropout(0.2),
        layers.Dense(1, activation="sigmoid"),
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss="binary_crossentropy",
        metrics=[keras.metrics.AUC(name="auc")],
    )
    return model

for lr in learning_rates:
    tf.keras.backend.clear_session()
    model = build_nn(X_train_bal.shape[1], lr)
    early_stop = keras.callbacks.EarlyStopping(
        monitor="val_auc",
        mode="max",
        patience=8,
        restore_best_weights=True,
    )
    history = model.fit(
        X_train_bal,
        y_train_bal,
        validation_data=(X_cv, y_cv),
        epochs=50,
        batch_size=32,
        verbose=0,
        callbacks=[early_stop],
    )
    cv_proba = model.predict(X_cv, verbose=0).ravel()
    cv_pred = (cv_proba >= 0.5).astype(int)
    metrics = {
        "learning_rate": lr,
        "accuracy": accuracy_score(y_cv, cv_pred),
        "precision": precision_score(y_cv, cv_pred, zero_division=0),
        "recall": recall_score(y_cv, cv_pred, zero_division=0),
        "f1": f1_score(y_cv, cv_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_cv, cv_proba),
        "epochs": len(history.history["loss"]),
    }
    results.append((metrics, model))

nn_results = pd.DataFrame([m for m, _ in results]).sort_values("roc_auc", ascending=False)
print("Neural-network validation results:")
print(nn_results)

best_nn_metrics, best_nn_model = max(results, key=lambda item: item[0]["roc_auc"])
print("\nBest learning rate:", best_nn_metrics["learning_rate"])
print("Best CV ROC-AUC:", best_nn_metrics["roc_auc"])


Neural-network validation results:
   learning_rate  accuracy  precision    recall        f1   roc_auc  epochs
1         0.0010  0.987055   0.984746  0.987976  0.986358  0.999419      50
4         0.0005  0.984155   0.976504  0.990380  0.983393  0.999183      50
0         0.0100  0.979080   0.993900  0.961740  0.977556  0.998718      40
3         0.0050  0.978977   0.970101  0.986008  0.977990  0.998718      17
2         0.0001  0.973488   0.965703  0.978793  0.972204  0.997084      50

Best learning rate: 0.001
Best CV ROC-AUC: 0.9994187154023382


In [14]:
def evaluate_model(name, y_true, y_proba):
    y_pred = (y_proba >= 0.5).astype(int)
    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_proba),
    }
    print(f"\n{name} evaluation:")
    for metric_name, metric_value in metrics.items():
        print(f"{metric_name}: {metric_value:.4f}")
    print("Classification report:")
    print(classification_report(y_true, y_pred, zero_division=0))
    print("Confusion matrix:\n", confusion_matrix(y_true, y_pred))
    return metrics

xgb_test_metrics = evaluate_model("XGBoost", y_test, xgb.predict_proba(X_test)[:, 1])
nn_test_proba = best_nn_model.predict(X_test, verbose=0).ravel()
nn_test_metrics = evaluate_model(f"Neural Network (lr={best_nn_metrics['learning_rate']})", y_test, nn_test_proba)

comparison = pd.DataFrame([
    {"model": "XGBoost", **xgb_test_metrics},
    {"model": f"Neural Network (lr={best_nn_metrics['learning_rate']})", **nn_test_metrics},
]).sort_values("roc_auc", ascending=False)

print("\nTest-set comparison:")
print(comparison)


XGBoost evaluation:
accuracy: 1.0000
precision: 1.0000
recall: 1.0000
f1: 1.0000
roc_auc: 1.0000
Classification report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      5083
           1       1.00      1.00      1.00      4574

    accuracy                           1.00      9657
   macro avg       1.00      1.00      1.00      9657
weighted avg       1.00      1.00      1.00      9657

Confusion matrix:
 [[5083    0]
 [   0 4574]]

Neural Network (lr=0.001) evaluation:
accuracy: 0.9906
precision: 0.9891
recall: 0.9910
f1: 0.9901
roc_auc: 0.9996
Classification report:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      5083
           1       0.99      0.99      0.99      4574

    accuracy                           0.99      9657
   macro avg       0.99      0.99      0.99      9657
weighted avg       0.99      0.99      0.99      9657

Confusion matrix:
 [[5033   50]
 [  41 4